In [ ]:
import os
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

PROJECT_ROOT = os.path.dirname(os.getcwd())
TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "kafka_bronze_clickstream")

builder = (
    SparkSession.builder.appName("Query_Bronze")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# %%
df = spark.read.format("delta").load(TARGET_PATH)
df.show(5, truncate=False)

In [ ]:
# Load Delta Table Reference and Record Count

df = spark.read.format("delta").load(TARGET_PATH)
delta_table = DeltaTable.forPath(spark, TARGET_PATH)

print(f"Total Bronze Records Ingested: {df.count():,}")

In [ ]:
# View Schema And Sample Records (Clickstream events replayer)

df.printSchema()
df.select("event_time", "event_type", "price", "_ingested_at", "_source_file").show(10, truncate=False)

In [ ]:
# Check Delta Commit History
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

In [ ]:
# Inspect Corrupt Payloads (DLQ Candidate Check)

from pyspark.sql.functions import col

# Inspect Corrupt Payloads (DLQ Candidate Check)
corrupt_df = df.filter(col("price") == "CORRUPT_PRICE_PAYLOAD")
print(f"Corrupt Records Count: {corrupt_df.count()}")
corrupt_df.select("event_time", "event_type", "price", "_source_file").show(10, truncate=False)

